# Notebook 53 — Preparación del corpus Silver

La capa Bronze refleja fielmente el archivo recibido. La capa Silver crea el contrato que
consumirán AI Search, el agente y MLflow:

- `chunk_id`: clave estable y determinística;
- `chunk_text`: texto que se convertirá en embedding;
- metadatos (`title`, `question_id`, versión y ruta fuente);
- etiquetas de evaluación (`question`, `expected_response`) que **no se sincronizarán al
  índice**

SQuAD ya trae contextos de tamaño de párrafo, ese va a ser el "chunk" a utilizar. La decisión queda explícita,
medida y validada.


## 1. Configuración


In [ ]:
import re
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "big_data_ii_2025", "1. Catálogo UC")
dbutils.widgets.text("esquema", "spark_examples", "2. Schema UC")
dbutils.widgets.text("volume", "agenteval_squadv2", "3. Volume UC")

CATALOG = dbutils.widgets.get("catalogo").strip()
SCHEMA = dbutils.widgets.get("esquema").strip()
VOLUME = dbutils.widgets.get("volume").strip()

for nombre, valor in {"catalogo": CATALOG, "esquema": SCHEMA, "volume": VOLUME}.items():
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", valor):
        raise ValueError(f"Identificador inválido en {nombre}: {valor!r}")

T_BRONZE = f"{CATALOG}.{SCHEMA}.agenteval_squadv2_bronze"
T_CORPUS = f"{CATALOG}.{SCHEMA}.agenteval_squadv2_corpus"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

assert spark.catalog.tableExists(T_BRONZE), (
    f"No existe {T_BRONZE}. Ejecuta primero el notebook 52."
)

print(f"Entrada : {T_BRONZE}")
print(f"Salida  : {T_CORPUS}")


## 2. Limpiar y crear una clave estable

No aplicamos `trim` a `context`: modificar un solo carácter invalidaría `answer_start`.
`chunk_id` se calcula con SHA-256 sobre título y contexto. El mismo documento produce el
mismo ID en cualquier reejecución.


In [ ]:
bronze = spark.table(T_BRONZE)

silver = (
    bronze
    .filter(
        F.col("question_id").isNotNull()
        & F.col("question").isNotNull()
        & F.col("context").isNotNull()
        & F.col("answer_text").isNotNull()
    )
    .dropDuplicates(["question_id"])
    .withColumn(
        "chunk_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("title"), F.lit("")),
                F.col("context"),
            ),
            256,
        ),
    )
    .select(
        "chunk_id",
        F.col("context").alias("chunk_text"),
        "title",
        "question_id",
        "question",
        F.col("answer_text").alias("expected_response"),
        "answer_start",
        F.length("context").alias("chunk_char_count"),
        F.size(F.split(F.trim("context"), r"\s+")).alias("chunk_word_count"),
        "source_version",
        "source_path",
        F.current_timestamp().alias("silver_updated_at"),
    )
)

display(silver.limit(10))


## 3. Validar el contrato del corpus

AI Search exige que la clave primaria sea única. Además comprobamos nuevamente el offset y
medimos el tamaño de los chunks. `databricks-gte-large-en` admite entradas mucho mayores que
estos párrafos, por lo que no necesitamos subdividirlos para esta demostración.


In [ ]:
n_rows = silver.count()
n_unique_chunks = silver.select("chunk_id").distinct().count()
n_unique_questions = silver.select("question_id").distinct().count()
n_bad_offsets = silver.filter(
    F.expr(
        "substring(chunk_text, answer_start + 1, length(expected_response)) "
        "<> expected_response"
    )
).count()

assert n_rows == 100, f"Se esperaban 100 filas Silver; se encontraron {n_rows}."
assert n_unique_chunks == n_rows, (
    "Hay contextos repetidos. Para un dataset general se separarían corpus y etiquetas; "
    "sampleqa100.json fue construido con 100 contextos distintos."
)
assert n_unique_questions == n_rows, "question_id dejó de ser único."
assert n_bad_offsets == 0, f"Hay {n_bad_offsets} offsets inválidos."

stats = silver.select(
    F.min("chunk_word_count").alias("min_palabras"),
    F.expr("percentile_approx(chunk_word_count, 0.5)").alias("mediana_palabras"),
    F.max("chunk_word_count").alias("max_palabras"),
    F.max("chunk_char_count").alias("max_caracteres"),
).first()

print(f"Filas / chunks únicos : {n_rows}")
print(f"Preguntas únicas      : {n_unique_questions}")
print(f"Palabras por chunk    : min={stats['min_palabras']}, "
      f"mediana={stats['mediana_palabras']}, max={stats['max_palabras']}")
print(f"Máximo de caracteres  : {stats['max_caracteres']}")
print("✓ Contrato Silver validado")


## 4. Escribir Delta Silver y habilitar Change Data Feed

El índice Delta Sync necesita Change Data Feed (CDF). Lo habilitamos aquí al crear la fuente
y el notebook 54 lo confirmará defensivamente antes de construir el índice.


In [ ]:
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(T_CORPUS)
)

spark.sql(
    f"""
    ALTER TABLE {T_CORPUS}
    SET TBLPROPERTIES (
      'delta.enableChangeDataFeed' = 'true',
      'layer' = 'silver',
      'quality.primary_key' = 'chunk_id',
      'rag.embedding_column' = 'chunk_text'
    )
    """
)

spark.sql(
    f"COMMENT ON TABLE {T_CORPUS} IS "
    "'Corpus Silver gobernado para AI Search y evaluación del agente RAG.'"
)

print(f"Tabla Silver creada: {T_CORPUS}")


## 5. Revisar propiedades y distribución

Las columnas `question` y `expected_response` permanecen en la tabla gobernada para evaluar
el ejercicio. El notebook 54 sincroniza al índice únicamente `chunk_id`, `chunk_text` y
`title`; por tanto, el generador nunca recibe la respuesta de referencia.


In [ ]:
props = {
    r["key"]: r["value"]
    for r in spark.sql(f"SHOW TBLPROPERTIES {T_CORPUS}").collect()
}
assert props.get("delta.enableChangeDataFeed", "false").lower() == "true"

print(f"CDF: {props['delta.enableChangeDataFeed']}")
display(
    spark.table(T_CORPUS)
    .groupBy("title")
    .count()
    .orderBy(F.desc("count"), "title")
)


## 6. Comprobación en Catalog Explorer

Abre `agenteval_squadv2_corpus` y verifica:

- 100 filas y 100 valores distintos de `chunk_id`;
- propiedad `delta.enableChangeDataFeed = true`;
- lineage desde la tabla Bronze después de ejecutar ambos notebooks;
- el historial Delta muestra una escritura idempotente.

Siguiente notebook: **54 — Índice administrado con Databricks AI Search**.
